In [ ]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

In [ ]:
path = "/content/drive/MyDrive/Sensor_DL/final_data"

X_train = np.load(os.path.join(path, "X_train.npy"))
y_train = np.load(os.path.join(path, "y_train.npy"))

X_val = np.load(os.path.join(path, "X_val.npy"))
y_val = np.load(os.path.join(path, "y_val.npy"))

X_test = np.load(os.path.join(path, "X_test.npy"))
y_test = np.load(os.path.join(path, "y_test.npy"))

In [ ]:
y_train -= 1
y_val -= 1
y_test -= 1

num_classes = len(np.unique(y_train))

In [ ]:
y_train = to_categorical(y_train, num_classes)
y_val = to_categorical(y_val, num_classes)
y_test = to_categorical(y_test, num_classes)

In [ ]:
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (24665, 50, 21) (24665, 12)
Val: (2700, 50, 21) (2700, 12)
Test: (6818, 50, 21) (6818, 12)


In [ ]:
model = Sequential([
    Input(shape=(50, X_train.shape[2])),

    # CNN Block
    Conv1D(64, 3, padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),

    Conv1D(128, 3, padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),

    # Optional feature compression
    Conv1D(128, 1, activation='relu'),

    #  Bidirectional LSTM
    Bidirectional(LSTM(64)),

    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 50, 64)         │         4,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 50, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 50, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 50, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 50, 128)        │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 153,932 (601.30 KB)

 Trainable params: 153,548 (599.80 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
checkpoint = ModelCheckpoint(
    filepath="/content/drive/MyDrive/Sensor_DL/best_cnn_bilstm.keras",
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1
)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=256,
    callbacks=[checkpoint, early_stop, lr_scheduler],
    verbose=1
)

Epoch 1/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1900 - loss: 2.2799
Epoch 1: val_accuracy improved from None to 0.53037, saving model to /content/drive/MyDrive/Sensor_DL/best_cnn_bilstm.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Sensor_DL/best_cnn_bilstm.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step - accuracy: 0.3229 - loss: 1.9319 - val_accuracy: 0.5304 - val_loss: 1.3705 - learning_rate: 0.0010
Epoch 2/30
95/97 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6479 - loss: 1.0110
Epoch 2: val_accuracy improved from 0.53037 to 0.79000, saving model to /content/drive/MyDrive/Sensor_DL/best_cnn_bilstm.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Sensor_DL/best_cnn_bilstm.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.6919 - loss: 0.8878 - val_accuracy: 0.7900 - val_loss: 0.6162 - learning_rate: 0.0010
Epoch 3/30
96/97 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7929 - loss: 0.6116
Epoch 3: val_accuracy improved fr

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(" CNN + BiLSTM Test Accuracy:", accuracy)

214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9512 - loss: 0.1555
 CNN + BiLSTM Test Accuracy: 0.9511587023735046
